# Parte 3 — Limpieza y control de calidad

Ejecuta este notebook desde el repositorio (`Ejecutar todas`). Lee `datos/crudos/mercadolibre_raw.csv`
—el CSV que produce la Parte 2— y escribe el dataset limpio, las tablas de control y la sección 4 del informe.

No vuelve a ejecutar el scraper ni consulta Mercado Libre. Conserva los valores ausentes; no inventa
precios anteriores ni vendedores. Solo necesita `pandas` (ver `requirements.txt`).

| Salida | Ruta |
| --- | --- |
| Dataset limpio | `datos/limpios/mercadolibre_limpio.csv` |
| Calidad por columna | `datos/calidad/calidad_por_columna.csv` |
| Resumen por categoría | `datos/calidad/resumen_por_categoria.csv` |
| Filas separadas | `datos/calidad/filas_para_revision.csv` |
| Repeticiones retiradas | `datos/calidad/duplicados_retirados.csv` |
| Precios excluidos | `datos/calidad/precios_excluidos.csv` |
| Sección 4 del informe | `documentos/Dataset_preliminar.md` |


In [1]:
from pathlib import Path
from urllib.parse import urlsplit, parse_qs
import re
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUTA = RAIZ / 'datos' / 'crudos' / 'mercadolibre_raw.csv'
LIMPIOS = RAIZ / 'datos' / 'limpios'
CALIDAD = RAIZ / 'datos' / 'calidad'
DOCUMENTOS = RAIZ / 'documentos'
for carpeta in (LIMPIOS, CALIDAD, DOCUMENTOS):
    carpeta.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RUTA, encoding='utf-8-sig', dtype='string')
df = raw.copy()
df.insert(0, 'fila_csv', range(2, len(df) + 2))
print('Archivo:', RUTA)
print('Dimensiones originales:', raw.shape)
print(raw.isna().mean().mul(100).round(2).to_string())

Archivo: C:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\datos\crudos\mercadolibre_raw.csv
Dimensiones originales: (5080, 13)
categoria                 0.00
pagina                    0.00
posicion_pagina           0.00
id_publicacion            0.00
titulo                    0.00
moneda                    0.00
precio_actual_texto       0.00
precio_anterior_texto    40.08
cuotas_texto              2.09
vendedor_texto           64.07
url_publicacion           0.00
url_fuente                0.00
fecha_extraccion          0.00


## Normalización e identificadores
Los precios usan punto para miles y coma para decimales: `S/ 4.849` se convierte en `4849.00` y `S/ 987,90` en `987.90`.

El identificador original puede proceder de `/p/MPE...`, mientras la misma URL contiene otro código en `wid`. Para identificar la oferta se prioriza `wid` y, si falta, el código de una ruta directa que no contenga `/p/`. Se mantiene el identificador del scraper como `id_extraido`. Las filas sin un identificador de oferta recuperable se separan para revisión; no se equipara automáticamente un código de catálogo a una publicación. Esta regla se basa en la estructura del CSV y no implica una verificación en vivo de las ofertas.

In [2]:
for c in raw.columns:
    df[c] = df[c].str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA)

def oferta_id(url):
    if pd.isna(url):
        return pd.NA
    u = urlsplit(url)
    parametros = parse_qs(u.query + '&' + u.fragment)
    wid = parametros.get('wid', [''])[0]
    if re.fullmatch(r'MPE[0-9]+', wid):
        return wid
    m = re.search(r'/MPE-?([0-9]+)(?:-|/|$)', u.path)
    if m and '/p/' not in u.path:
        return 'MPE' + m.group(1)
    return pd.NA

def precio(serie):
    s = serie.str.replace(r'^S/\s*', '', regex=True).str.replace(' ', '', regex=False)
    valido = s.str.fullmatch(r'(?:[0-9]{1,3}(?:\.[0-9]{3})+|[0-9]+)(?:,[0-9]{1,2})?', na=False)
    return pd.to_numeric(s.where(valido).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')

df = df.rename(columns={'id_publicacion': 'id_extraido'})
df['id_publicacion'] = df['url_publicacion'].map(oferta_id).astype('string')
df['precio_actual'] = precio(df['precio_actual_texto'])
df['moneda'] = df['moneda'].replace({'S/': 'PEN'})
df['numero_cuotas'] = pd.to_numeric(df['cuotas_texto'].str.extract(r'^(\d+) cuotas? de ', expand=False), errors='coerce').astype('Int64')
importe = df['cuotas_texto'].str.extract(r'cuotas? de (S/ [\d., ]+)', expand=False).str.strip()
df['monto_cuota'] = precio(importe)
for c in ['pagina', 'posicion_pagina']:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
df['fecha_extraccion'] = pd.to_datetime(df['fecha_extraccion'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
print('IDs originales diferentes del ID de oferta:', int((df.id_extraido != df.id_publicacion).fillna(False).sum()))
print('Filas sin ID de oferta recuperable:', int(df.id_publicacion.isna().sum()))
print('Precios actuales no interpretables:', int(df.precio_actual.isna().sum()))

IDs originales diferentes del ID de oferta: 2599
Filas sin ID de oferta recuperable: 56
Precios actuales no interpretables: 0


## Calidad y duplicados
Se separan registros sin campos esenciales válidos. Para ofertas repetidas se conserva la observación más reciente y, en empates, la última fila del archivo. No se combinan atributos de observaciones distintas. El listado es una muestra de resultados de búsquedas; `categoria` representa la búsqueda de origen, que puede incluir accesorios u otros productos.

Las cuotas ausentes permanecen nulas. Un texto ausente no demuestra ausencia de financiamiento. Tampoco se asume que un precio anterior ausente sea igual al precio actual.

In [3]:
invalidas = (df['id_publicacion'].isna() | df['titulo'].isna() | df['categoria'].isna()
             | df['precio_actual'].isna() | df['precio_actual'].le(0)
             | ~df['moneda'].eq('PEN') | df['fecha_extraccion'].isna()
             | df['pagina'].isna() | df['posicion_pagina'].isna())
revision = df.loc[invalidas].copy()
revision.to_csv(CALIDAD / 'filas_para_revision.csv', index=False, encoding='utf-8')
validas = df.loc[~invalidas].sort_values(['fecha_extraccion', 'fila_csv'], kind='stable')
repetidas = validas.duplicated('id_publicacion', keep='last')
validas.loc[repetidas].to_csv(CALIDAD / 'duplicados_retirados.csv', index=False, encoding='utf-8')
unicas = validas.loc[~repetidas].sort_values('fila_csv').copy()
print('Filas para revisión:', len(revision))
print('Repeticiones retiradas:', int(repetidas.sum()))
print('Ofertas únicas antes de exclusiones:', len(unicas))

Filas para revisión: 56
Repeticiones retiradas: 605
Ofertas únicas antes de exclusiones: 4419


## Precios cuestionados
Siete ofertas se excluyen por un precio incoherente con el producto anunciado, revisadas una por una y confirmadas en el CSV crudo: no provienen de la conversión numérica. Son exclusiones puntuales de esta muestra, no un umbral general: no se elimina ningún registro solo por ser caro o barato, y los demás valores extremos se conservan. La lista es explícita para que la decisión quede auditable; la evidencia se guarda en `precios_excluidos.csv`, sin verificar el precio real ni imputar reemplazos.

In [4]:
MOTIVO = ('Precio cuestionado por incoherencia con el producto; confirmado en el CSV crudo. '
          'Exclusión para análisis, sin verificación del precio real ni imputación.')
IDS_EXCLUIDOS = [
    'MPE824658294',    # Xiaomi Redmi Note 13 Pro+ a S/ 1.250.000
    'MPE959377798',    # Lenovo V15 a S/ 29.000
    'MPE1090653346',   # Televisor Daewoo a S/ 99.999
    'MPE1094203286',   # Televisor Daewoo a S/ 10
    'MPE1094202682',   # Televisor Daewoo a S/ 10
    'MPE1090833326',   # Refrigerador Daewoo a S/ 999.999
    'MPE1092274734',   # Refrigeradora Indurama a S/ 2
]

marca_excluidos = unicas['id_publicacion'].isin(IDS_EXCLUIDOS)
excluidos = unicas.loc[marca_excluidos].assign(motivo_exclusion=MOTIVO)
assert len(excluidos) == len(IDS_EXCLUIDOS), 'Algún identificador excluido no está en el dataset.'
excluidos.to_csv(CALIDAD / 'precios_excluidos.csv', index=False, encoding='utf-8')
unicas = unicas.loc[~marca_excluidos]

# Conservamos atributos con cobertura suficiente. Los textos crudos quedan en el CSV original.
columnas = ['id_publicacion', 'id_extraido', 'categoria', 'titulo', 'moneda',
            'precio_actual', 'numero_cuotas', 'monto_cuota', 'cuotas_texto',
            'pagina', 'posicion_pagina', 'url_publicacion', 'url_fuente', 'fecha_extraccion']
limpio = unicas[columnas].reset_index(drop=True)
calidad = pd.DataFrame({'variable': limpio.columns,
    'tipo_pandas': [str(limpio[c].dtype) for c in limpio],
    'nulos': [int(limpio[c].isna().sum()) for c in limpio],
    'porcentaje_nulos': [limpio[c].isna().mean() * 100 for c in limpio],
    'valores_distintos': [limpio[c].nunique() for c in limpio]})
print('Filas originales:', len(raw))
print('Precios excluidos:', len(excluidos))
print('Dimensiones finales:', limpio.shape)
print(calidad.to_string(index=False))

Filas originales: 5080
Precios excluidos: 7
Dimensiones finales: (4412, 14)
        variable    tipo_pandas  nulos  porcentaje_nulos  valores_distintos
  id_publicacion         string      0            0.0000               4412
     id_extraido         string      0            0.0000               4412
       categoria         string      0            0.0000                  5
          titulo         string      0            0.0000               4181
          moneda         string      0            0.0000                  1
   precio_actual        Float64      0            0.0000               1952
   numero_cuotas          Int64     55            1.2466                  2
     monto_cuota        Float64     55            1.2466               2225
    cuotas_texto         string     55            1.2466               2335
          pagina          Int64      0            0.0000                 20
 posicion_pagina          Int64      0            0.0000                 60
 url_publica

## Verificación y exportación
El dataset principal mantiene más de seis atributos, variables numéricas y categóricas, al menos mil ofertas identificadas y como máximo 10 % de nulos por columna. Las columnas de página y posición son metadatos; no se presentan como características comerciales del producto.

In [5]:
assert len(limpio) >= 1000, 'No se alcanza el mínimo de registros.'
assert len(limpio.columns) >= 6
assert limpio.id_publicacion.is_unique
assert calidad.porcentaje_nulos.le(10).all(), 'Hay columnas que superan el 10 % de nulos.'
assert limpio.precio_actual.gt(0).all()
assert limpio.numero_cuotas.dropna().gt(0).all()
assert limpio.monto_cuota.dropna().gt(0).all()
assert len(raw) == len(revision) + int(repetidas.sum()) + len(excluidos) + len(limpio)
limpio.to_csv(LIMPIOS / 'mercadolibre_limpio.csv', index=False, encoding='utf-8', date_format='%Y-%m-%d %H:%M:%S')
calidad.to_csv(CALIDAD / 'calidad_por_columna.csv', index=False, encoding='utf-8')
releido = pd.read_csv(LIMPIOS / 'mercadolibre_limpio.csv', encoding='utf-8')
assert releido.shape == limpio.shape
assert releido.id_publicacion.is_unique
assert releido.isna().mean().le(.10).all()
resumen = limpio.groupby('categoria').agg(registros=('id_publicacion', 'size'),
    precio_minimo=('precio_actual', 'min'), precio_mediano=('precio_actual', 'median'),
    precio_promedio=('precio_actual', 'mean'), precio_maximo=('precio_actual', 'max')).round(2)
resumen.to_csv(CALIDAD / 'resumen_por_categoria.csv', encoding='utf-8')
print(resumen.to_string())
print('Validaciones aprobadas y CSV reabierto correctamente.')

                registros  precio_minimo  precio_mediano  precio_promedio  precio_maximo
categoria                                                                               
audifonos             959           16.9           138.0           376.08         8959.0
celulares             937          38.61          1632.0          2075.04         9498.0
laptops               928          459.0          4898.5          5807.42        24665.0
refrigeradoras        646           52.0          1314.0          1805.08        20741.0
televisores           942           25.0          1289.5          1907.65        59999.0
Validaciones aprobadas y CSV reabierto correctamente.


## Texto de la sección Dataset preliminar
La siguiente celda escribe `documentos/Dataset_preliminar.md` con los resultados reales de esta ejecución: es la sección 4 del informe. `documentos/notas_integracion.md` registra qué deben ajustar las otras partes.

In [6]:
def tabla(frame):
    encabezado = '| ' + ' | '.join(map(str, frame.columns)) + ' |'
    separador = '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |'
    filas = ['| ' + ' | '.join(str(x).replace('|', '/') for x in fila) + ' |' for fila in frame.itertuples(index=False, name=None)]
    return '\n'.join([encabezado, separador] + filas)

texto = f"""# 4. Dataset preliminar

## 4.1 Descripción de la muestra
Se procesaron {len(raw):,} observaciones crudas de Mercado Libre Perú, obtenidas entre {df.fecha_extraccion.min()} y {df.fecha_extraccion.max()}, según las marcas temporales del scraper. El archivo no registra zona horaria. Las búsquedas de origen fueron celulares, laptops, televisores, refrigeradoras y audifonos. El dataset final contiene {len(limpio):,} ofertas con identificadores únicos y {len(limpio.columns)} columnas.

La unidad de observación es la oferta identificada en la URL, conservando su última aparición en el archivo. La muestra procede de páginas de resultados y no es aleatoria ni representativa de todo el mercado peruano. La categoría es la búsqueda de origen y puede incluir accesorios o productos relacionados. Los precios describen ofertas, no transacciones realizadas.

## 4.2 Preparación y control de calidad
Se normalizaron espacios, se transformaron precios a formato numérico, se estandarizó la moneda a PEN y se convirtieron las fechas a datetime. Se extrajeron el número de cuotas y el monto por cuota de los textos observados, manteniendo los nulos cuando la información no estaba disponible.

El identificador extraído originalmente por el scraper se conserva como id_extraido. Para id_publicacion se tomó el parámetro wid de la URL o, cuando estaba ausente, el código de una ruta directa sin /p/. Se separaron {len(revision)} filas sin campos esenciales válidos y se retiraron {int(repetidas.sum())} observaciones repetidas, conservando la última por fecha y orden de archivo. Adicionalmente se excluyeron {len(excluidos)} ofertas con precios cuestionados, identificadas por código y precio: Xiaomi Redmi Note 13 Pro+ a S/ 1.250.000, refrigerador Daewoo a S/ 999.999, televisor Daewoo a S/ 99.999, refrigeradora Indurama a S/ 2, dos televisores Daewoo a S/ 10 y Lenovo V15 a S/ 29.000. Son exclusiones puntuales acordadas para esta muestra, no límites generales aplicables a todos los productos. Estos importes ya estaban en el CSV crudo; no fueron generados por la conversión numérica. La exclusión es una decisión de calidad para el análisis, sin verificar el precio real ni imputar reemplazos. Los registros separados se guardaron como evidencia para revisión.

No se encontraron duplicados de fila completa en el CSV original ({int(raw.duplicated().sum())}); las repeticiones se detectaron por identificador de oferta, aunque sus posiciones, fechas o enlaces de seguimiento fueran diferentes.

Se excluyeron precio_anterior_texto y vendedor_texto del dataset principal debido a sus ausencias originales de {raw.precio_anterior_texto.isna().mean()*100:.2f} % y {raw.vendedor_texto.isna().mean()*100:.2f} %, respectivamente. Ambas columnas permanecen en el CSV crudo. No se imputaron precios anteriores, vendedores ni descuentos. La falta de un precio anterior no permite concluir que el descuento sea cero.

## 4.3 Variables y cobertura
Las variables comerciales incluyen titulo, categoria, moneda, precio_actual, numero_cuotas, monto_cuota y cuotas_texto. Los identificadores, páginas, posiciones, URL y fecha proporcionan trazabilidad. El dataset incluye variables numéricas y categóricas, supera los 1000 registros y los seis atributos mínimos, y ninguna columna supera el 10 % de valores nulos. La tabla presenta los tipos en pandas antes de exportar; CSV no conserva tipos y deben declararse al volver a cargarlo.

{tabla(calidad.round(2))}

## 4.4 Estadísticos descriptivos
Todos los precios se expresan en soles (PEN). Tras las exclusiones puntuales descritas, se conservaron los demás valores extremos porque su magnitud por sí sola no prueba un error. Las estadísticas siguientes excluyen los siete casos documentados en precios_excluidos.csv. No se ha verificado el precio real de cada publicación ni se afirma que todos los precios restantes sean correctos. Las diferencias entre categorías no equivalen a diferencias entre productos comparables.

{tabla(resumen.reset_index())}

## 4.5 Vista previa
Se muestra una selección de columnas de las primeras cinco filas. El CSV contiene la totalidad de los atributos descritos.

{tabla(limpio[['id_publicacion', 'categoria', 'titulo', 'precio_actual', 'numero_cuotas']].head())}

## 4.6 Limitaciones
No se extrajeron calificación, número de opiniones, condición, marca ni envío gratuito. Por tanto, estos datos no permiten analizar esas características ni la reputación del vendedor. Tampoco permiten calcular descuentos de manera completa. La deduplicación conserva una sola observación por oferta y no constituye una serie histórica. La identidad se recuperó de las URL guardadas y no se verificó mediante consultas adicionales al sitio.
"""
(DOCUMENTOS / 'Dataset_preliminar.md').write_text(texto, encoding='utf-8')
print('Sección 4 del informe guardada en', DOCUMENTOS / 'Dataset_preliminar.md')

Sección 4 del informe guardada en C:\Users\yared\Documents\ACD\Proyecto\proyectoacd_mercadolibre\documentos\Dataset_preliminar.md


## Resultado
La Parte 3 termina aquí: el dataset limpio y las tablas de control quedan versionados en el repositorio y
`documentos/Dataset_preliminar.md` se copia al informe. `filas_para_revision.csv`, `duplicados_retirados.csv`
y `precios_excluidos.csv` son trazabilidad: no deben volver a mezclarse con el CSV limpio.